# Análise de Distância e Agrupamento (Clustering) de Clientes

Este notebook tem como objetivo segmentar clientes com base em seu histórico de compras (Recência, Frequência e Valor). Utilizaremos o algoritmo de **Agrupamento Hierárquico (Método de Ward)** para encontrar clientes com comportamentos semelhantes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Pré-processamento e Transformação
from sklearn.preprocessing import PowerTransformer

# Algoritmos de Clusterização e Distâncias
from scipy.spatial import distance
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# Configurações de estilo para os gráficos
sns.set_theme(style="whitegrid")

## 1. Carga e Preparação dos Dados
Vamos carregar a base de vendas (`fato_vendas.csv`) e agregar as informações para o nível de cliente, criando uma visão `dim_clientes`.

In [ ]:
# Carregando os dados de vendas
df_vendas = pd.read_csv('bases/fato_vendas.csv')

# Agrupando por cliente para gerar as variáveis Recência, Frequência e Valor (Monetário)
# Obs: Pode adaptar a query abaixo se você já tiver a tabela dim_clientes pronta
dim_clientes = df_vendas.groupby('id_cliente').agg(
    recencia_dias=('dias_atras_compra', 'min'), # Dias desde a última compra
    frequencia=('id_venda', 'count'),           # Total de compras feitas
    valor_venda=('valor_venda', 'sum')          # Valor total gasto
).reset_index()

dim_clientes.head()

## 2. Normalização dos Dados (Feature Scaling)
Como variáveis como `recencia_dias` possuem valores atípicos (outliers) e assimetria, e `frequencia` está em uma escala muito menor, **é obrigatório normalizar** os dados para que todos tenham o mesmo peso no algoritmo de distância.

Vamos usar o `PowerTransformer` para aproximar as distribuições de uma curva normal.

In [ ]:
# Selecionando apenas as variáveis quantitativas para o cluster
features = ['recencia_dias', 'frequencia', 'valor_venda']
dados_cluster = dim_clientes[features]

# Aplicando o PowerTransformer em TODAS as features simultaneamente
pt = PowerTransformer()
clientes_pad = pt.fit_transform(dados_cluster)

# Convertendo de volta para DataFrame apenas para facilitar a visualização (opcional)
clientes_pad_df = pd.DataFrame(clientes_pad, columns=features)
clientes_pad_df.head()

## 3. Agrupamento Hierárquico (Método de Ward)
O método de Ward une os grupos de forma a minimizar o aumento da variância (distância) em cada passo. Vamos criar a matriz de ligação (linkage) e visualizar a árvore gerada (Dendrograma).

In [ ]:
# Criando a matriz de ligação usando o método de Ward
matriz_ward = linkage(clientes_pad, method='ward')

# Desenhando o Dendrograma
plt.figure(figsize=(12, 6))
plt.title('Dendrograma de Clientes (Método de Ward)')
plt.xlabel('Índice dos Clientes')
plt.ylabel('Distância (Aumento de Variância)')

# truncate_mode='level' ajuda a visualizar melhor limitando a profundidade da árvore desenhada
dendrogram(matriz_ward, truncate_mode='level', p=5) 
plt.show()

## 4. Definição e Atribuição dos Clusters
Neste passo, faremos o corte manual do dendrograma. Vamos manter a definição de 3 clusters usando `criterion='maxclust'`. O algoritmo descerá pelo dendrograma até formar exatamente 3 grandes grupos.

In [ ]:
# Cortando a árvore para formar exatamente 3 clusters
dim_clientes['cluster_designado'] = fcluster(matriz_ward, t=3, criterion='maxclust')

# Analisando o tamanho de cada cluster formado
distribuicao_clusters = dim_clientes['cluster_designado'].value_counts().sort_index()
print("Distribuição de clientes por cluster:\n", distribuicao_clusters)

# Analisando a média de cada variável por cluster para entender o PERFIL de cada grupo
perfil_clusters = dim_clientes.groupby('cluster_designado')[features].mean()
display(perfil_clusters)